# 🎬 Audience Ratings Behaviour — Notebook 7
## Data Cleaning, Merging & Behavioural Feature Engineering

**Series:** May Newsletter — Movie Intelligence (Part 3 of 3)  
**Prerequisites:** `movies_clean.parquet` from NB1, plus `ratings_small.csv` and `links.csv`

### The join chain
```
ratings_small  ──(movieId)──▶  links  ──(tmdbId → id)──▶  movies_clean
```
`ratings_small` uses MovieLens IDs. `links` is the bridge table mapping those IDs to TMDB IDs,
which is the key used in `movies_clean`.

### What we do here
1. Load all three source files and harmonise join keys
2. Aggregate the 100k individual ratings into per-film audience metrics
3. Engineer the four behavioural features that drive the newsletter story:
   - `avg_user_rating` — mean audience score
   - `rating_std` — disagreement / polarisation score
   - `rating_count` — audience reach
   - `commercial_vs_audience_gap` — the delta between box office rank and rating rank
4. Classify every film into one of four quadrants: **Blockbuster**, **Hidden Gem**,
   **Overhyped**, or **Critical Failure**
5. Save `audience_clean.parquet` for NB8 and NB9


## 0 · Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)


pandas : 2.2.2
numpy  : 2.0.2


## 1 · Load Source Files

In [2]:
movies  = pd.read_parquet("movies_clean.parquet")
ratings = pd.read_csv("ratings_small.csv")
links   = pd.read_csv("links.csv")

# Harmonise join keys — everything to string
movies["id"]     = movies["id"].astype(str).str.strip()
links["tmdbId"]  = pd.to_numeric(links["tmdbId"], errors="coerce")
links["tmdbId"]  = links["tmdbId"].dropna().astype(int).astype(str)
links["movieId"] = links["movieId"].astype(str)
ratings["movieId"] = ratings["movieId"].astype(str)

print(f"movies_clean    : {movies.shape}")
print(f"ratings_small   : {ratings.shape}  |  {ratings['userId'].nunique()} users  |  {ratings['movieId'].nunique():,} unique films")
print(f"links           : {links.shape}    |  {links['tmdbId'].notna().sum():,} rows with tmdbId")


movies_clean    : (5128, 23)
ratings_small   : (100004, 4)  |  671 users  |  9,066 unique films
links           : (45843, 3)    |  45,624 rows with tmdbId


## 2 · Aggregate Ratings to Film Level

In [3]:
film_ratings = (
    ratings.groupby("movieId")
    .agg(
        avg_user_rating = ("rating", "mean"),
        rating_std      = ("rating", "std"),
        rating_count    = ("rating", "count"),
        rating_median   = ("rating", "median"),
        pct_5star       = ("rating", lambda x: (x == 5.0).mean() * 100),
        pct_1star       = ("rating", lambda x: (x <= 1.0).mean() * 100),
    )
    .reset_index()
)

# rating_std is NaN for films with only 1 rating — replace with 0
film_ratings["rating_std"] = film_ratings["rating_std"].fillna(0)

print(f"Films with aggregated ratings : {len(film_ratings):,}")
print()
print("Rating distribution overview:")
print(film_ratings[["avg_user_rating","rating_std","rating_count"]].describe().round(3).to_string())


Films with aggregated ratings : 9,066

Rating distribution overview:
       avg_user_rating  rating_std  rating_count
count         9066.000    9066.000      9066.000
mean             3.292       0.579        11.031
std              0.882       0.526        24.051
min              0.500       0.000         1.000
25%              2.844       0.000         1.000
50%              3.500       0.672         3.000
75%              3.966       0.966         9.000
max              5.000       3.182       341.000


## 3 · Bridge via Links Table

In [4]:
# Step 1: attach tmdbId to the aggregated ratings
ratings_linked = film_ratings.merge(
    links[["movieId","tmdbId"]].dropna(subset=["tmdbId"]),
    on="movieId", how="inner"
)

print(f"After links merge : {len(ratings_linked):,} films")

# Step 2: merge onto movies_clean using tmdbId → id
df_audience = movies.merge(
    ratings_linked.rename(columns={"tmdbId": "id"}),
    on="id", how="inner"
)

print(f"After movies merge: {len(df_audience):,} films  (overlap with financial universe)")
print()
print("Coverage:")
print(f"  movies_clean universe : {len(movies):,} films")
print(f"  films with user ratings: {len(df_audience):,}  ({len(df_audience)/len(movies)*100:.1f}%)")


After links merge : 9,048 films
After movies merge: 3,810 films  (overlap with financial universe)

Coverage:
  movies_clean universe : 5,128 films
  films with user ratings: 3,810  (74.3%)


## 4 · Filter for Analysis Quality

In [5]:
# Require at least 10 user ratings for stable aggregates
MIN_RATINGS = 10
df_audience = df_audience[df_audience["rating_count"] >= MIN_RATINGS].copy()

print(f"Films with ≥{MIN_RATINGS} user ratings : {len(df_audience):,}")
print()
print("Final column set:")
print(df_audience.columns.tolist())


Films with ≥10 user ratings : 1,657

Final column set:
['id', 'imdb_id', 'title', 'release_date', 'release_year', 'release_month', 'month_name', 'decade', 'budget', 'revenue', 'profit', 'ROI', 'ROI_capped', 'budget_tier', 'vote_average', 'vote_count', 'popularity', 'runtime', 'primary_genre', 'genre_list', 'genre_count', 'original_language', 'production_countries', 'movieId', 'avg_user_rating', 'rating_std', 'rating_count', 'rating_median', 'pct_5star', 'pct_1star']


## 5 · Engineer the Commercial vs Audience Gap

In [6]:
# Percentile-rank each film on both revenue and audience rating
# Gap > 0 means the film earns more at the box office than audiences reward it
df_audience["revenue_pct_rank"] = df_audience["revenue"].rank(pct=True) * 100
df_audience["rating_pct_rank"]  = df_audience["avg_user_rating"].rank(pct=True) * 100

df_audience["comm_vs_audience_gap"] = (
    df_audience["revenue_pct_rank"] - df_audience["rating_pct_rank"]
)

print("Commercial vs Audience Gap statistics:")
print(df_audience["comm_vs_audience_gap"].describe().round(2).to_string())
print()
print("Interpretation:")
print("  Large positive gap → commercially successful but audience-disappointing (Overhyped)")
print("  Large negative gap → low revenue but audience favourite (Hidden Gem)")


Commercial vs Audience Gap statistics:
count    1657.00
mean       -0.00
std        44.32
min       -99.16
25%       -32.95
50%         2.84
75%        34.10
max        93.30

Interpretation:
  Large positive gap → commercially successful but audience-disappointing (Overhyped)
  Large negative gap → low revenue but audience favourite (Hidden Gem)


## 6 · Quadrant Classification

In [7]:
# Split on median revenue and median audience rating for clean quadrant cuts
rev_med  = df_audience["revenue"].median()
rat_med  = df_audience["avg_user_rating"].median()

def classify_quadrant(row):
    high_rev = row["revenue"]        >= rev_med
    high_rat = row["avg_user_rating"] >= rat_med
    if   high_rev and high_rat:  return "Blockbuster"
    elif high_rev and not high_rat: return "Overhyped"
    elif not high_rev and high_rat: return "Hidden Gem"
    else:                           return "Critical Failure"

df_audience["quadrant"] = df_audience.apply(classify_quadrant, axis=1)

print("Quadrant distribution:")
print(df_audience["quadrant"].value_counts().to_string())
print()
print(f"  Revenue median  : ${rev_med/1e6:.1f}M")
print(f"  Rating median   : {rat_med:.2f} / 5.0")


Quadrant distribution:
quadrant
Hidden Gem          485
Overhyped           484
Blockbuster         345
Critical Failure    343

  Revenue median  : $88.0M
  Rating median   : 3.52 / 5.0


## 7 · Polarisation Tier

In [9]:
std_75 = df_audience["rating_std"].quantile(0.75)
std_25 = df_audience["rating_std"].quantile(0.25)

def polarisation_tier(s):
    if s >= std_75: return "Divisive"
    if s <= std_25: return "Consensus"
    return "Mixed"

df_audience["polarisation"] = df_audience["rating_std"].apply(polarisation_tier)

print("Polarisation distribution:")
print(df_audience["polarisation"].value_counts().to_string())
print(f"  Std dev 25th pct : {std_25:.2f}")
print(f"  Std dev 75th pct : {std_75:.2f}")

Polarisation distribution:
polarisation
Mixed        827
Consensus    415
Divisive     415
  Std dev 25th pct : 0.81
  Std dev 75th pct : 1.05


## 8 · Sanity-Check Sample

In [10]:
cols = ["title","release_year","revenue","ROI","avg_user_rating",
        "rating_count","rating_std","quadrant","polarisation","primary_genre"]

print("=== Blockbusters (sample) ===")
print(df_audience[df_audience["quadrant"]=="Blockbuster"].nlargest(5,"revenue")[cols].to_string(index=False))
print()
print("=== Hidden Gems (sample) ===")
print(df_audience[df_audience["quadrant"]=="Hidden Gem"].nlargest(5,"avg_user_rating")[cols].to_string(index=False))
print()
print("=== Most Overhyped (sample) ===")
print(df_audience[df_audience["quadrant"]=="Overhyped"].nlargest(5,"comm_vs_audience_gap")[cols].to_string(index=False))


=== Blockbusters (sample) ===
                                       title  release_year      revenue       ROI  avg_user_rating  rating_count  rating_std    quadrant polarisation   primary_genre
                                      Avatar        2009.0 2787965087.0 10.763566         3.641791            67    1.025456 Blockbuster        Mixed          Action
                                The Avengers        2012.0 1519557910.0  5.907081         4.010870            46    0.884993 Blockbuster        Mixed Science Fiction
                     Avengers: Age of Ultron        2015.0 1405403694.0  4.019299         3.884615            13    0.711625 Blockbuster    Consensus          Action
Harry Potter and the Deathly Hallows: Part 2        2011.0 1342000000.0  9.736000         4.220588            34    0.605420 Blockbuster    Consensus          Family
                                      Frozen        2013.0 1274219009.0  7.494793         4.041667            12    0.582250 Blockbuster    

## 9 · Save

In [11]:
df_audience.to_parquet("audience_clean.parquet", index=False)

print(f"Saved audience_clean.parquet — {len(df_audience):,} rows × {df_audience.shape[1]} columns")
print()
print("Key engineered columns:")
for col in ["avg_user_rating","rating_std","rating_count","rating_median",
            "revenue_pct_rank","rating_pct_rank","comm_vs_audience_gap",
            "quadrant","polarisation"]:
    print(f"  ✔  {col}")
print()
print("Proceed to Notebook 8 → Audience Behaviour Analysis ▶")


Saved audience_clean.parquet — 1,657 rows × 35 columns

Key engineered columns:
  ✔  avg_user_rating
  ✔  rating_std
  ✔  rating_count
  ✔  rating_median
  ✔  revenue_pct_rank
  ✔  rating_pct_rank
  ✔  comm_vs_audience_gap
  ✔  quadrant
  ✔  polarisation

Proceed to Notebook 8 → Audience Behaviour Analysis ▶
